In [1]:
import matplotlib.pyplot as plt          #########
import networkx as nx
from torch_geometric.utils.convert import to_networkx
import torch
from torch_geometric.data import Dataset
from torch_geometric.data import download_url
import os
from torch_geometric.io import read_planetoid_data
from torch_geometric.datasets import Planetoid
import numpy as np
from torch_geometric.data import Data
from torch.nn import Linear
import numpy as np
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv,SAGEConv,GATConv
import os.path as osp   #
from torch_geometric.nn import global_mean_pool
import scipy.sparse as sp
import torch_geometric.nn as pyg_nn
from torch_geometric.data import DataLoader
import warnings
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.metrics import matthews_corrcoef
warnings.filterwarnings("ignore", category=Warning)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr
from tqdm import tqdm
import torch.nn as nn
from config import config_runtime
from collections import Counter
from sklearn.metrics import confusion_matrix
import shutil

In [ ]:
path='/Glycan_binding/test_npz'
interface_rec_list = [f[0:10] for f in os.listdir(path) ]
interface_rec_list.sort()

In [ ]:
for num2,numm in enumerate(interface_rec_list):
    pdb_name=f'{numm}.npz'
    #if pdb_name !='7LD9_1_A_B.npz':
    #    continue 
    pdb_name2=pdb_name[0:4]
    pdb_name4=pdb_name[0:10]
    A_chain=pdb_name4[-1]
    B_chain=pdb_name4[-3]
    pdb_name5=pdb_name[0:6]
    pdb_name3=f'{pdb_name4}_npz'
    # 源文件路径
    source = f"/Glycan_binding/test_npz/{pdb_name}"
    base_path='/Glycan_binding/test/tu'
    os.makedirs(os.path.join(base_path, pdb_name3), exist_ok=True)
    # 目标路径（可以是文件夹，也可以是完整的新文件路径）
    destination = f"/Glycan_binding/test/tu/{pdb_name3}/{pdb_name}"

    # 移动文件
    shutil.copy(source, destination)

    print(f"{pdb_name3}文件移动成功！")

    
    
        #test
    file1=f'/Glycan_binding/test/tu/{pdb_name3}'

    class MyOwnDataset2(Dataset):
        def __init__(self, root, transform=None, pre_transform=None):
            super().__init__(root, transform, pre_transform)
            

        # 返回数据集源文件名，告诉原始的数据集存放在哪个文件夹下面，如果数据集已经存放进去了，那么就会直接从raw文件夹中读取。
        @property
        def raw_file_names(self):
            # pass # 不能使用pass，会报join() argument must be str or bytes, not 'NoneType'错误
            return []

        # 首先寻找processed_paths[0]路径下的文件名也就是之前process方法保存的文件名
        @property
        def processed_file_names(self):
            return ['datas0.pt','datas1.pt']

        # 用于从网上下载数据集，下载原始数据到指定的文件夹下，自己的数据集可以跳过
        def download(self):
            pass
        def process(self):
            #
            data_list=[]
            for i in range(1):
                path_list=os.listdir(file1)
                path_list.sort()
                npz=path_list[i]
                #print(npz)
                #pdb2=path_list[i]
                aa=os.path.join(file1,npz)
        
                fileload=np.load(aa)  #读取npz文件
                key=list(fileload.keys()) #获取npz里面的键，由一个个字典组成
                #print(key)

                node=fileload['H'] #获取节点以及节点特征，数据类型为numpy.ndarray
                #node = torch.tensor(x) #数据类型为torch.Tensor
                #print(np.shape(node))

                adj=fileload['A1'] #data.edge_index节点和边的邻接矩阵
                #adj2=fileload['A2'] #data.edge_attr: 边属性
        

                #邻接矩阵转换成COO稀疏矩阵及转换  
                edge_index_temp = sp.coo_matrix(adj)  
                #print('edge_index_temp为：')
                #print(np.shape(edge_index_temp))
                tar=fileload['T']
        
        
                indices = np.vstack((edge_index_temp.row, edge_index_temp.col))
                edge_index = torch.LongTensor(indices)
                #print(np.shape(edge_index))
                #节点及节点特征数据转换
                x = node
                #print(x)
                #x = x.squeeze(0)
                x = torch.FloatTensor(x)
                #print(np.shape(x))
                #print(tar)
                a=tar
                y=torch.LongTensor(a)
                #print(y)
        
                #edge_attr=adj2

                #构建数据集:为一张图，节点数量，节点特征，Coo稀疏矩阵的边(邻接矩阵)，边的特征矩阵,一个图一个标签
                data=Data(x=x, edge_index=edge_index,y=y)
                if self.pre_filter is not None and not self.pre_filter(data):
                    continue
                if self.pre_transform is not None:
                    data = self.pre_transform(data)
                torch.save(data,osp.join(self.processed_dir,'datas{}.pt'.format(i)))
        
        def len(self):
            return 1

        def get(self, idx):
            data = torch.load(osp.join(self.processed_dir, 'datas{}.pt'.format(idx)))
            return data
        
    dataset_test=MyOwnDataset2(f'/Glycan_binding/test/tu/{pdb_name3}/{pdb_name4}_pt')


    class ImprovedTripleGraphModel(nn.Module):
        def __init__(self, num_node_features, num_output_features=2, dropout_rate=0.3):
            super(ImprovedTripleGraphModel, self).__init__()
            self.dropout = nn.Dropout(dropout_rate)

            # 定义三层GATConv用于B链图
            self.conv_1 = SAGEConv(num_node_features, 512)
            self.conv_2 = SAGEConv(512, 1024)
            self.conv_3 = SAGEConv(1024, 2)

            # 最终层（不应用sigmoid）
            self.fc1 = nn.Linear(256, 512)  # 3个64维的特征向量融合后的维度
            self.fc2 = nn.Linear(512, num_output_features)  # 输出层
            
            
        def forward(self, x,edge_index, batch):
            # 处理B链图
            
            
            x = F.dropout(x, p=0.5, training=self.training)
            x = self.conv_1(x,edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=0.5, training=self.training)
            x = self.conv_2(x,edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=0.5, training=self.training)
            x = self.conv_3(x,edge_index)
            x = F.relu(x)
            return F.softmax(x, dim=1)
            
    model_path='glycan_binding.pt'
    model=ImprovedTripleGraphModel(1310)
    state_dict = torch.load(model_path)
    model.load_state_dict(state_dict)
    model.eval()
    
    test_loader = DataLoader(dataset_test, batch_size=1, shuffle=False)
    device=config_runtime['device']
    model.to(device)
    
    preds=[]
    tru=[]
    for data in test_loader:
        data=data.to(device)
        out = model(data.x, data.edge_index, data.batch)
        
        pred=out.argmax(dim=1)
        aa = data.y
            
            # 收集真实标签和预测标签
        preds.extend(pred.cpu().numpy())
        tru.extend(aa.cpu().numpy())
        
    rew_all_sasa_A=f'/Glycan_binding/test/{pdb_name4}/A/ppi/sasa/{pdb_name4}.pdb'
    rew_all_sasa_B=f'/Glycan_binding/test/{pdb_name4}/B/ppi/sasa/{pdb_name4}.pdb'
    
    all_A=[]
    with open(rew_all_sasa_A,'r') as rec_file:
            line = rec_file.readlines()
            for i in line:
                if i[0:3] =='SEQ':
                    i=i.strip()
                    chain=i[4]
                    bb=i[12:15]         #残基名称
                    dd=i[6:11].strip()   #残基编号
                    cc=float(i[19:26])   #sasa
                #print(bb,cc)            # 将文件中的SASA值存储到字典中  
                    all_A.append((chain,dd,bb))
                    
    all_B=[]
    with open(rew_all_sasa_B,'r') as rec_file:
            line = rec_file.readlines()
            for i in line:
                if i[0:3] =='SEQ':
                    i=i.strip()
                    chain=i[4]
                    bb=i[12:15]         #残基名称
                    dd=i[6:11].strip()   #残基编号
                    cc=float(i[19:26])   #sasa
                #print(bb,cc)            # 将文件中的SASA值存储到字典中  
                    all_B.append((chain,dd,bb))
                    
                    
    all=all_A+all_B
    
    result_dict={value:item for item,value in zip(preds,all)}
    
    new_resiude=[]
    for key,value in result_dict.items():
        if str(value)=='1':
            new_resiude.append(key)
            
    input_pdb=f'/Glycan_binding/test/{pdb_name4}/all/{pdb_name4}.pdb'
    output_pdb=f'/Glycan_binding/test/tu/{pdb_name3}/{pdb_name4}_color.pdb'
    
    
    with open(input_pdb,'r') as infile,open(output_pdb,'w') as outfile:
        for line in infile:
            if line.startswith('ATOM'):
                atom_name = line[12:16].strip()  
                residue_name = line[17:20].strip()  
                chain_id = line[21].strip()  
                residue_number = (line[22:27].strip())  
                x = float(line[30:38].strip())  
                y = float(line[38:46].strip())  
                z = float(line[46:54].strip()) 
                line2=line[60:66].strip()
                if (chain_id,residue_number,residue_name) in new_resiude:

                    b_factor_value = 1
                else:
                    b_factor_value = 0
                        # 修改 B-factor 列（第 61-66 列）
                line = line[:60] + f"{b_factor_value:6.2f}" + line[66:]
            if line.startswith('HETATM'):
                b_factor_value = 0
                line = line[:60] + f"{b_factor_value:6.2f}" + line[66:]
            outfile.write(line)      

1B41_1_B_A_npz文件移动成功！


Processing...
Done!


1G9M_1_C_G_npz文件移动成功！
1GC1_1_C_G_npz文件移动成功！


Processing...
Done!
Processing...
Done!
Processing...


1I3R_1_E_F_npz文件移动成功！


...

Done!
Processing...
Done!


8X53_1_C_A_npz文件移动成功！
8X54_1_C_A_npz文件移动成功！


Processing...
Done!
Processing...
Done!
Processing...
Done!


9AVV_1_F_A_npz文件移动成功！
9AVV_1_G_C_npz文件移动成功！


Processing...
Done!
